# Pocket Retrieval via Set Transformer + Contrastive Learning — Prototype

Notebook prototype. Sections:
1. Imports & config
2. Atom featurization (RDKit basic, d_atom=36)
3. GIN ligand encoder (vanilla / +virtual node)
4. PMA / SAB / MAB blocks
5. Cluster encoder (SAB → SAB → PMA_k=8)
6. Projection head + MoCo queue
7. InfoNCE loss with queue
8. Dummy data generator (mimics pdbbind_final_groups.csv grouping)
9. Smoke test: forward + backward + queue update


## 1. Imports & config

In [3]:
import math, random, copy
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, GINEConv
from torch_geometric.data import Data, Batch
from torch_geometric.utils import to_undirected

from rdkit import Chem
from rdkit.Chem import AllChem

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

@dataclass
class Config:
    d_atom: int = 37
    d_bond: int = 6
    d_h: int = 128
    d_proj: int = 64
    gin_layers: int = 3
    heads: int = 4
    k_prototypes: int = 8
    queue_size: int = 4096
    momentum: float = 0.999
    temperature: float = 0.07
    proto_collapse: str = "mean"   # "mean" | "max"
    gin_variant: str = "vanilla"   # "vanilla" | "vnode"

cfg = Config()
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:2


## 2. Atom featurization (RDKit basic, d_atom = 36)

In [4]:
ATOMIC_NUM_LIST = [6, 7, 8, 9, 15, 16, 17, 35, 53]   # C,N,O,F,P,S,Cl,Br,I
DEGREE_LIST = [0,1,2,3,4,5,6]
FORMAL_CHARGE_LIST = [-2,-1,0,1,2]
HYB_LIST = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]
NUMHS_LIST = [0,1,2,3,4]

def one_hot(val, allowed):
    vec = [0]*(len(allowed)+1)
    if val in allowed:
        vec[allowed.index(val)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom: Chem.Atom) -> List[int]:
    feats = []
    feats += one_hot(atom.GetAtomicNum(), ATOMIC_NUM_LIST)   # 10
    feats += one_hot(atom.GetDegree(), DEGREE_LIST)          #  8 -> need 7 per spec, but +unk = 8
    feats += one_hot(atom.GetFormalCharge(), FORMAL_CHARGE_LIST)  # 6
    feats += one_hot(atom.GetHybridization(), HYB_LIST)      # 6
    feats += [int(atom.GetIsAromatic())]                     # 1
    feats += one_hot(atom.GetTotalNumHs(), NUMHS_LIST)       # 6
    return feats

# probe length
_probe = Chem.MolFromSmiles("CCO")
_d_atom = len(atom_features(_probe.GetAtomWithIdx(0)))
print("d_atom =", _d_atom)
cfg.d_atom = _d_atom

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]
def bond_features(bond: Chem.Bond) -> List[int]:
    feats = []
    feats += one_hot(bond.GetBondType(), BOND_TYPES)   # 5
    feats += [int(bond.GetIsConjugated())]             # 1
    return feats
_probe2 = Chem.MolFromSmiles("C=C")
_d_bond = len(bond_features(_probe2.GetBondWithIdx(0)))
print("d_bond =", _d_bond)
cfg.d_bond = _d_bond

def mol_to_data(mol: Chem.Mol) -> Data:
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index = []
    edge_attr = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        edge_index += [[i,j],[j,i]]
        bf = bond_features(b)
        edge_attr += [bf, bf]
    if len(edge_index) == 0:
        edge_index = torch.zeros((2,0), dtype=torch.long)
        edge_attr  = torch.zeros((0, cfg.d_bond), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

d = mol_to_data(Chem.MolFromSmiles("c1ccccc1O"))
print("sample Data:", d, " x.shape:", d.x.shape, " edge_index.shape:", d.edge_index.shape)


d_atom = 37
d_bond = 6
sample Data: Data(x=[7, 37], edge_index=[2, 14], edge_attr=[14, 6])  x.shape: torch.Size([7, 37])  edge_index.shape: torch.Size([2, 14])


## 3. GIN ligand encoder

Two variants:
- `vanilla`: standard GIN (Xu et al. 2019) — uses `GINConv` on atom features only, bonds collapsed to topology.
- `vnode`:    GIN + virtual node — adds one global node connected to all atoms, message-passes through it each layer.

After GIN layers, atom embeddings `H_atoms ∈ R^(N × d_h)` are produced.


In [5]:
def mlp(d_in, d_h, d_out):
    return nn.Sequential(nn.Linear(d_in, d_h), nn.GELU(), nn.Linear(d_h, d_out))

class GINLigandEncoder(nn.Module):
    def __init__(self, cfg: Config, variant: str = "vanilla"):
        super().__init__()
        self.cfg = cfg
        self.variant = variant
        self.atom_proj = nn.Linear(cfg.d_atom, cfg.d_h)
        self.convs = nn.ModuleList([
            GINConv(mlp(cfg.d_h, cfg.d_h, cfg.d_h), train_eps=True)
            for _ in range(cfg.gin_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(cfg.d_h) for _ in range(cfg.gin_layers)])
        if variant == "vnode":
            self.vnode_emb = nn.Parameter(torch.zeros(1, cfg.d_h))
            self.vnode_mlps = nn.ModuleList([
                mlp(cfg.d_h, cfg.d_h, cfg.d_h) for _ in range(cfg.gin_layers)
            ])
        elif variant != "vanilla":
            raise ValueError(variant)

    def forward(self, data: Batch) -> Tuple[torch.Tensor, torch.Tensor]:
        # returns (atom_embeddings [ΣN, d_h], batch_vec [ΣN])
        x = self.atom_proj(data.x)
        batch = data.batch
        B = int(batch.max().item()) + 1 if batch.numel() > 0 else 1
        if self.variant == "vnode":
            vn = self.vnode_emb.expand(B, -1).contiguous()  # [B, d_h]
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            if self.variant == "vnode":
                x = x + vn[batch]
            x = conv(x, data.edge_index)
            x = norm(x)
            x = F.gelu(x)
            if self.variant == "vnode":
                # aggregate atoms to vnode
                agg = torch.zeros_like(vn).index_add_(0, batch, x)
                counts = torch.bincount(batch, minlength=B).clamp(min=1).unsqueeze(-1).float()
                vn = vn + self.vnode_mlps[i](agg / counts)
        return x, batch


## 4. MAB / SAB / PMA blocks (Set Transformer primitives)

- `MAB(X, Y) = LayerNorm(H + FF(H))`,  `H = LayerNorm(X + MHA(X, Y, Y))`
- `SAB(X) = MAB(X, X)`
- `PMA_k(Z) = MAB(S_k, Z)`, `S_k` learnable seeds.

These accept padded `[B, N, d]` tensors with an optional `mask [B, N]` (True = real, False = pad).


In [6]:
class MAB(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.mha = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.ff  = nn.Sequential(nn.Linear(d, 2*d), nn.GELU(), nn.Linear(2*d, d))

    def forward(self, X, Y, key_padding_mask=None):
        # X: [B, Nq, d]  Y: [B, Nk, d]  key_padding_mask: [B, Nk] True=pad
        h, _ = self.mha(X, Y, Y, key_padding_mask=key_padding_mask, need_weights=False)
        h = self.ln1(X + h)
        out = self.ln2(h + self.ff(h))
        return out

class SAB(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.mab = MAB(d, heads)
    def forward(self, X, mask=None):
        # mask: [B, N] True = real ; convert to key_padding_mask True=pad
        kpm = (~mask) if mask is not None else None
        return self.mab(X, X, key_padding_mask=kpm)

class PMA(nn.Module):
    def __init__(self, d, heads, k):
        super().__init__()
        self.S = nn.Parameter(torch.randn(1, k, d) * 0.02)
        self.mab = MAB(d, heads)
    def forward(self, Z, mask=None):
        # Z: [B, N, d]
        B = Z.size(0)
        S = self.S.expand(B, -1, -1)
        kpm = (~mask) if mask is not None else None
        return self.mab(S, Z, key_padding_mask=kpm)


## 5. From GIN output to ligand vector, and ligand-set to prototype

- Atoms (flat) → `PMA_1` → ligand vector `z_lig ∈ R^d_h`. Implemented by padding atoms per ligand into [B_lig, N_max, d_h].
- Ligands in a cluster → SAB → SAB → PMA_k → `P_cluster ∈ R^(k, d_h)`.


In [7]:
def flat_to_padded(x_flat, batch, B=None):
    """x_flat [ΣN, d], batch [ΣN] -> (padded [B, N_max, d], mask [B, N_max])"""
    if B is None:
        B = int(batch.max().item()) + 1
    counts = torch.bincount(batch, minlength=B)
    N_max = int(counts.max().item())
    d = x_flat.size(-1)
    padded = x_flat.new_zeros(B, N_max, d)
    mask   = torch.zeros(B, N_max, dtype=torch.bool, device=x_flat.device)
    # scatter per-graph positions
    pos_in_graph = torch.zeros_like(batch)
    # compute pos_in_graph: cumulative count within batch index
    # vectorized:
    order = torch.argsort(batch, stable=True)
    sorted_batch = batch[order]
    # within each group, positions are 0..count-1
    # build via arange offsets
    starts = torch.zeros(B, dtype=torch.long, device=batch.device)
    starts[1:] = torch.cumsum(counts, dim=0)[:-1]
    pos_sorted = torch.arange(batch.numel(), device=batch.device) - starts[sorted_batch]
    pos_in_graph[order] = pos_sorted
    padded[batch, pos_in_graph] = x_flat
    mask[batch, pos_in_graph] = True
    return padded, mask

class LigandPooler(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.pma = PMA(cfg.d_h, cfg.heads, k=1)
    def forward(self, atom_emb, batch, B):
        padded, mask = flat_to_padded(atom_emb, batch, B=B)
        z = self.pma(padded, mask=mask)        # [B, 1, d_h]
        return z.squeeze(1)                    # [B, d_h]

class ClusterEncoder(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.sab1 = SAB(cfg.d_h, cfg.heads)
        self.sab2 = SAB(cfg.d_h, cfg.heads)
        self.pma  = PMA(cfg.d_h, cfg.heads, k=cfg.k_prototypes)
    def forward(self, Z_cluster, mask=None):
        # Z_cluster: [B_clusters, M_max, d_h] ; mask: [B_clusters, M_max]
        h = self.sab1(Z_cluster, mask=mask)
        h = self.sab2(h, mask=mask)
        return self.pma(h, mask=mask)   # [B_clusters, k, d_h]


## 6. Projection head + MoCo queue

Projection: `Linear → GELU → Linear → L2-normalize`. Applied per-token.

Prototype collapse: `mean` over k tokens (fixed) or `max` over k tokens (each query matched to best token). Switchable via `cfg.proto_collapse`.

MoCo queue: ring buffer of L2-normalized prototype vectors (shape `[Q, d_proj]`), updated FIFO each step. EMA "key encoder" is a deep-copy of the main encoders updated with momentum `m`.


In [8]:
class ProjectionHead(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_h, cfg.d_h), nn.GELU(), nn.Linear(cfg.d_h, cfg.d_proj)
        )
    def forward(self, x):
        x = self.net(x)
        return F.normalize(x, dim=-1)

def collapse_prototypes(P_proj, query_proj=None, mode="mean"):
    """
    P_proj: [B_c, k, d_proj]  (L2-normalized per token)
    query_proj: [B_q, d_proj] (needed for mode='max' to compute per-query best token; for retrieval logits)
    Returns:
        if mode == 'mean': p_c [B_c, d_proj] (re-normalized)
        if mode == 'max':  returns P_proj unchanged ([B_c, k, d_proj]); scoring done elsewhere
    """
    if mode == "mean":
        p = P_proj.mean(dim=1)
        return F.normalize(p, dim=-1)
    elif mode == "max":
        return P_proj
    else:
        raise ValueError(mode)

class MoCoQueue:
    def __init__(self, size, dim, device):
        self.size = size
        self.dim = dim
        self.buf = F.normalize(torch.randn(size, dim, device=device), dim=-1)
        self.ptr = 0
        self.full = False
    @torch.no_grad()
    def enqueue(self, keys):
        # keys: [B, dim] — collapsed (single vector) prototypes from key encoder
        n = keys.size(0)
        if n >= self.size:
            self.buf = keys[-self.size:].detach().clone()
            self.ptr = 0
            self.full = True
            return
        end = self.ptr + n
        if end <= self.size:
            self.buf[self.ptr:end] = keys.detach()
        else:
            first = self.size - self.ptr
            self.buf[self.ptr:] = keys[:first].detach()
            self.buf[:n-first]  = keys[first:].detach()
            self.full = True
        self.ptr = end % self.size
        if end >= self.size:
            self.full = True
    def get(self):
        if self.full:
            return self.buf
        return self.buf[:self.ptr] if self.ptr > 0 else self.buf[:0]


## 7. InfoNCE with queue (mean and max-sim variants)

Anchor `q_i` (query ligand projection, `[d_proj]`).
Positive prototype `p_i⁺` from leave-one-out cluster encoding.
Negatives: other in-batch prototypes + queue prototypes.

For `mean` mode: scoring = `q · p_c`, p_c collapsed to single vector.
For `max` mode: scoring per cluster = `max_t q · p_c[t]`. Queue still stores single collapsed vector (mean) — keeping queue simple. Ablation can extend.


In [9]:
def info_nce_loss(q, p_pos, p_neg_inbatch, queue, tau, mode="mean"):
    """
    q:               [B, d_proj]
    p_pos (mean):    [B, d_proj]            OR
    p_pos (max):     [B, k, d_proj]
    p_neg_inbatch:   [B-1 per anchor]; we approximate by using all B clusters (in-batch),
                     standard MoCo style: positives at index i, others as negatives.
                     -> pass full p_inbatch [B, d_proj] (mean) or [B, k, d_proj] (max).
    queue:           [Q, d_proj] (mean-collapsed prototypes)
    Returns scalar loss.
    """
    B = q.size(0)
    if mode == "mean":
        # in-batch logits: [B, B]
        logits_in = q @ p_pos.t()           # contains positives on diag
        if queue.numel() > 0:
            logits_q = q @ queue.t()        # [B, Q]
            logits = torch.cat([logits_in, logits_q], dim=1) / tau
        else:
            logits = logits_in / tau
        labels = torch.arange(B, device=q.device)
        return F.cross_entropy(logits, labels)
    elif mode == "max":
        # p_pos: [B, k, d_proj]   q: [B, d_proj]
        # in-batch: for query i vs cluster j: max_t q_i · p_j[t]
        # compute [B, B, k] then max over k
        sims = torch.einsum("id,jkd->ijk", q, p_pos)   # [B, B, k]
        logits_in = sims.max(dim=-1).values            # [B, B]
        if queue.numel() > 0:
            logits_q = q @ queue.t()
            logits = torch.cat([logits_in, logits_q], dim=1) / tau
        else:
            logits = logits_in / tau
        labels = torch.arange(B, device=q.device)
        return F.cross_entropy(logits, labels)
    else:
        raise ValueError(mode)


## 8. Top-level model wrapping all pieces + key (EMA) encoder


In [10]:
class PocketRetrievalModel(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ligand_enc  = GINLigandEncoder(cfg, variant=cfg.gin_variant)
        self.ligand_pool = LigandPooler(cfg)
        self.cluster_enc = ClusterEncoder(cfg)
        self.proj_query  = ProjectionHead(cfg)
        self.proj_proto  = ProjectionHead(cfg)

    def encode_ligands(self, batch: Batch):
        atom_emb, b = self.ligand_enc(batch)
        B = int(batch.batch.max().item()) + 1
        z = self.ligand_pool(atom_emb, b, B=B)    # [B_lig, d_h]
        return z

    def encode_cluster_from_ligands(self, z_ligs, cluster_ids):
        """z_ligs [L, d_h], cluster_ids [L] -> P [C, k, d_h] and order of unique ids."""
        unique = torch.unique(cluster_ids, sorted=True)
        C = unique.numel()
        counts = torch.tensor([(cluster_ids == u).sum() for u in unique], device=z_ligs.device)
        M_max = int(counts.max().item())
        padded = z_ligs.new_zeros(C, M_max, z_ligs.size(-1))
        mask = torch.zeros(C, M_max, dtype=torch.bool, device=z_ligs.device)
        for ci, u in enumerate(unique):
            sel = (cluster_ids == u).nonzero(as_tuple=True)[0]
            padded[ci, :sel.numel()] = z_ligs[sel]
            mask[ci, :sel.numel()] = True
        P = self.cluster_enc(padded, mask=mask)   # [C, k, d_h]
        return P, unique


## 9. Dummy data generator

Mimics `pdbbind_final_groups.csv` grouping: each row has `(ligand_smiles, pocket_cluster_id)`.
For the smoke test:
- Generate `C=6` clusters
- Each cluster has 3–5 ligands; ligands within a cluster share a scaffold (analogue series) — emulates "similar binders" — but with random substitutions.
- One query is sampled per cluster; the rest form the positive prototype.


In [11]:
SCAFFOLDS = [
    "c1ccc(cc1)C(=O)N",          # benzamide
    "c1ncccc1N",                 # aminopyridine
    "O=C(O)c1ccc(O)cc1",         # salicylic
    "N#Cc1ccc(N)cc1",            # benzonitrile-amine
    "c1ccc2[nH]ccc2c1",          # indole
    "c1ccc(cc1)S(=O)(=O)N",      # sulfonamide
]
SUBS = ["C", "CC", "OC", "Cl", "F", "CN", "CCN", "CO", "CCO"]

def random_analog(scaffold, rng):
    sub = rng.choice(SUBS)
    smi = scaffold + sub
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return Chem.MolFromSmiles(scaffold)
    return mol

def make_dummy_dataset(n_clusters=6, ligs_per_cluster=(3,5), seed=0):
    rng = random.Random(seed)
    rows = []  # list of dicts: smiles, pocket_cluster_id
    for ci, scaf in enumerate(SCAFFOLDS[:n_clusters]):
        M = rng.randint(*ligs_per_cluster)
        for _ in range(M):
            mol = random_analog(scaf, rng)
            rows.append({"smiles": Chem.MolToSmiles(mol), "pocket_cluster_id": ci})
    return rows

dummy = make_dummy_dataset()
for r in dummy[:5]:
    print(r)
print("total ligs:", len(dummy))


{'smiles': 'NCCNC(=O)c1ccccc1', 'pocket_cluster_id': 0}
{'smiles': 'CNC(=O)c1ccccc1', 'pocket_cluster_id': 0}
{'smiles': 'O=C(NF)c1ccccc1', 'pocket_cluster_id': 0}
{'smiles': 'O=C(NCCO)c1ccccc1', 'pocket_cluster_id': 0}
{'smiles': 'NCCNc1cccnc1', 'pocket_cluster_id': 1}
total ligs: 25


## 10. Smoke test

End-to-end:
1. Build dummy dataset (≥ 2 ligands per cluster).
2. For each cluster: pick one ligand as query; remaining as positive set.
3. Encode all ligands (query + positives) with one GIN forward pass over a single PyG `Batch`.
4. Build cluster prototypes from positives (key encoder for queue updates).
5. Run InfoNCE for both `mean` and `max` modes.
6. Backward pass; EMA update; enqueue keys.

The assertions check: shapes, finite loss, gradient flow, queue state.


In [12]:
def split_query_positives(rows):
    """For each cluster with M >= 2: return query indices, positive indices, cluster ids per ligand."""
    by_cluster: Dict[int, List[int]] = {}
    for i, r in enumerate(rows):
        by_cluster.setdefault(r["pocket_cluster_id"], []).append(i)
    queries, positives, q_cluster_ids = [], [], []   # positives = list of lists
    rng = random.Random(0)
    for cid, idxs in by_cluster.items():
        if len(idxs) < 2:
            continue
        q = rng.choice(idxs)
        pos = [i for i in idxs if i != q]
        queries.append(q)
        positives.append(pos)
        q_cluster_ids.append(cid)
    return queries, positives, q_cluster_ids

def build_pyg_batch(rows, indices, device):
    datas = []
    for i in indices:
        mol = Chem.MolFromSmiles(rows[i]["smiles"])
        datas.append(mol_to_data(mol))
    return Batch.from_data_list(datas).to(device)

def smoke_test(cfg: Config):
    model = PocketRetrievalModel(cfg).to(device)
    key_model = copy.deepcopy(model).to(device)
    for p in key_model.parameters():
        p.requires_grad_(False)
    queue = MoCoQueue(cfg.queue_size, cfg.d_proj, device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

    rows = make_dummy_dataset(n_clusters=6, ligs_per_cluster=(3,5), seed=1)
    queries, positives, q_cluster_ids = split_query_positives(rows)
    B = len(queries)
    print(f"B (clusters w/ query): {B}")

    # ---- query forward (online encoder)
    q_batch = build_pyg_batch(rows, queries, device)
    z_q = model.encode_ligands(q_batch)                # [B, d_h]
    q_proj = model.proj_query(z_q)                     # [B, d_proj]
    assert q_proj.shape == (B, cfg.d_proj)
    print("q_proj:", q_proj.shape)

    # ---- positive prototypes (key encoder, no grad)
    with torch.no_grad():
        pos_flat_idx, pos_cluster_local = [], []
        for ci, pos in enumerate(positives):
            for i in pos:
                pos_flat_idx.append(i)
                pos_cluster_local.append(ci)
        p_batch = build_pyg_batch(rows, pos_flat_idx, device)
        z_p = key_model.encode_ligands(p_batch)        # [L_pos, d_h]
        cluster_ids = torch.tensor(pos_cluster_local, device=device)
        P, order = key_model.encode_cluster_from_ligands(z_p, cluster_ids)   # [B, k, d_h]
        assert order.tolist() == list(range(B)), "cluster ordering mismatch"
        # project all k tokens
        P_proj = key_model.proj_proto(P)               # [B, k, d_proj]  L2-normed per token
        P_mean = F.normalize(P_proj.mean(dim=1), dim=-1)
    print("P_proj:", P_proj.shape, "  P_mean:", P_mean.shape)

    # ---- losses both modes
    loss_mean = info_nce_loss(q_proj, P_mean, None, queue.get(), cfg.temperature, mode="mean")
    loss_max  = info_nce_loss(q_proj, P_proj, None, queue.get(), cfg.temperature, mode="max")
    print(f"loss_mean = {loss_mean.item():.4f}   loss_max = {loss_max.item():.4f}")
    assert torch.isfinite(loss_mean) and torch.isfinite(loss_max)

    # ---- backward on mean variant (default)
    opt.zero_grad()
    loss_mean.backward()
    # check grads
    grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
    print(f"#params w/ grad: {len(grad_norms)}, mean grad norm: {np.mean(grad_norms):.4e}")
    assert len(grad_norms) > 0 and np.mean(grad_norms) > 0
    opt.step()

    # ---- EMA update
    with torch.no_grad():
        for pq, pk in zip(model.parameters(), key_model.parameters()):
            pk.data.mul_(cfg.momentum).add_(pq.data, alpha=1.0 - cfg.momentum)

    # ---- enqueue keys (use P_mean from key encoder)
    queue.enqueue(P_mean)
    print(f"queue ptr={queue.ptr}, full={queue.full}, current size={queue.get().shape}")

    return loss_mean.item(), loss_max.item()

print("=== vanilla GIN, mean collapse ===")
cfg_v = Config(gin_variant="vanilla", proto_collapse="mean")
smoke_test(cfg_v)

print("\n=== virtual-node GIN, max collapse ===")
cfg_b = Config(gin_variant="vnode", proto_collapse="max")
smoke_test(cfg_b)


=== vanilla GIN, mean collapse ===
B (clusters w/ query): 6
q_proj: torch.Size([6, 64])
P_proj: torch.Size([6, 8, 64])   P_mean: torch.Size([6, 64])
loss_mean = 1.9647   loss_max = 1.9607
#params w/ grad: 40, mean grad norm: 1.2029e+00
queue ptr=6, full=False, current size=torch.Size([6, 64])

=== virtual-node GIN, max collapse ===
B (clusters w/ query): 6
q_proj: torch.Size([6, 64])
P_proj: torch.Size([6, 8, 64])   P_mean: torch.Size([6, 64])
loss_mean = 1.8377   loss_max = 1.8407
#params w/ grad: 49, mean grad norm: 2.8520e-01
queue ptr=6, full=False, current size=torch.Size([6, 64])


(1.8377176523208618, 1.8406623601913452)

## 11. Shape & invariant sanity checks


In [13]:
# Permutation invariance check: cluster encoder output should be invariant to ligand order.
def test_cluster_permutation_invariance():
    cfg = Config()
    model = PocketRetrievalModel(cfg).to(device)
    rows = make_dummy_dataset(n_clusters=3, ligs_per_cluster=(4,4), seed=2)
    # take cluster 0
    idxs = [i for i, r in enumerate(rows) if r["pocket_cluster_id"] == 0]
    b1 = build_pyg_batch(rows, idxs, device)
    z1 = model.encode_ligands(b1)
    P1, _ = model.encode_cluster_from_ligands(z1, torch.zeros(len(idxs), dtype=torch.long, device=device))
    # shuffle
    perm = torch.randperm(len(idxs))
    idxs_p = [idxs[i] for i in perm.tolist()]
    b2 = build_pyg_batch(rows, idxs_p, device)
    z2 = model.encode_ligands(b2)
    P2, _ = model.encode_cluster_from_ligands(z2, torch.zeros(len(idxs), dtype=torch.long, device=device))
    # PMA seeds are ordered, so P1 and P2 token-wise should match (within attention precision)
    diff = (P1 - P2).abs().max().item()
    print(f"max |P1 - P2| under ligand permutation: {diff:.3e}")
    assert diff < 1e-4, "Cluster encoder is not permutation-invariant"

test_cluster_permutation_invariance()

# Atom permutation invariance for ligand pooler (PMA output should be invariant to atom order)
def test_ligand_permutation_invariance():
    cfg = Config()
    model = PocketRetrievalModel(cfg).to(device).eval()
    mol = Chem.MolFromSmiles("c1ccc(cc1)C(=O)NCO")
    d1 = mol_to_data(mol)
    # permute atoms by reordering x rows and remapping edge_index
    n = d1.x.size(0)
    perm = torch.randperm(n)
    inv = torch.argsort(perm)
    x2 = d1.x[perm]
    ei2 = inv[d1.edge_index]
    d2 = Data(x=x2, edge_index=ei2, edge_attr=d1.edge_attr)
    b1 = Batch.from_data_list([d1]).to(device)
    b2 = Batch.from_data_list([d2]).to(device)
    with torch.no_grad():
        z1 = model.encode_ligands(b1)
        z2 = model.encode_ligands(b2)
    diff = (z1 - z2).abs().max().item()
    print(f"max |z_lig(perm) - z_lig| : {diff:.3e}")
    assert diff < 1e-4, "Ligand pooler not permutation-invariant"

test_ligand_permutation_invariance()

print("\nALL CHECKS PASSED")


max |P1 - P2| under ligand permutation: 7.153e-07
max |z_lig(perm) - z_lig| : 5.960e-07

ALL CHECKS PASSED


---
# Part B — Real-data pipeline

Implements:
1. CSV loader for `pdbbind_final_groups.csv`
2. Time-based split (`release_year` ≤ 2018 train/val, 2019 test) at **cluster level**
3. SDF reader for `aligned_ligand_sdf`
4. Hard-negative augmentation (same `uniprot_id`, different `pocket_cluster_id`)
5. Training loop + Recall@K evaluation


## 12. CSV loader

Filter rules:
- `final_status == 'ok'`
- non-null `aligned_ligand_sdf`, `pocket_cluster_id`, `release_year`, `uniprot_id`
- one row per `pdb_id`


In [83]:
import os
import pandas as pd
from pathlib import Path

CSV_PATH_DEFAULT = "../pdbbind_final_groups.csv"
PDBBIND_ROOT_DEFAULT = "."   # base directory under which aligned_ligand_sdf paths resolve

def load_pdbbind_groups(csv_path: str = CSV_PATH_DEFAULT,
                        pdbbind_root: str = PDBBIND_ROOT_DEFAULT) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    n0 = len(df)
    df = df[df["final_status"] == "ok"].copy()
    print(df.columns)
    df = df.dropna(subset=["ligand_sdf", "pocket_cluster_id",
                           "release_year", "uniprot_id", "pdb_id"])
    df = df.drop_duplicates(subset=["pdb_id"])
    df["release_year"] = df["release_year"].astype(int)
    df["aligned_ligand_sdf_abs"] = df["ligand_sdf"].apply(
        lambda p: str(Path(pdbbind_root) / p)
    )
    print(f"loaded {n0} rows -> {len(df)} after filtering")
    print(f"unique pocket_cluster_id: {df['pocket_cluster_id'].nunique()}")
    print(f"unique uniprot_id:        {df['uniprot_id'].nunique()}")
    return df.reset_index(drop=True)


## 13. Time-based split at cluster level

Per the GroupBind paper convention: training/validation = ≤ 2018, test = 2019.
A pocket cluster is fully assigned to one split, with rule:
- If **all** ligands in a cluster are ≤ 2018  → train
- If **all** ligands in a cluster are == 2019 → test
- Mixed clusters are dropped (avoids any cluster spanning the split boundary).

Validation is a held-out 10% of train clusters (random).


In [84]:
def time_split_clusters(df: pd.DataFrame, val_frac: float = 0.1, seed: int = 0):
    cluster_years = df.groupby("pocket_cluster_id")["release_year"].agg(["min", "max"])
    train_full, test, mixed = [], [], []
    for cid, row in cluster_years.iterrows():
        if row["max"] <= 2018:
            train_full.append(cid)
        elif row["min"] >= 2019:
            test.append(cid)
        else:
            mixed.append(cid)
    rng = random.Random(seed)
    rng.shuffle(train_full)
    n_val = max(1, int(len(train_full) * val_frac))
    val = train_full[:n_val]
    train = train_full[n_val:]
    print(f"clusters: train={len(train)}  val={len(val)}  test={len(test)}  dropped(mixed)={len(mixed)}")
    return set(train), set(val), set(test)

def filter_clusters_with_min_size(df: pd.DataFrame, cluster_ids: set, min_size: int = 2) -> set:
    """Anchors require leave-one-out positive => cluster needs >= 2 ligands."""
    sizes = df.groupby("pocket_cluster_id").size()
    return {c for c in cluster_ids if sizes.get(c, 0) >= min_size}


## 14. SDF reader

`aligned_ligand_sdf` points to an `.sdf` file with the docked ligand (1 molecule).
Falls back gracefully if RDKit can't parse — that row is dropped at load time.


In [85]:
def read_ligand_from_sdf(sdf_path: str) -> Optional[Chem.Mol]:
    if not os.path.isfile(sdf_path):
        return None

    suppl = Chem.SDMolSupplier(
        sdf_path,
        removeHs=True,
        sanitize=False,
        strictParsing=False,
    )

    for mol in suppl:
        if mol is not None and mol.GetNumAtoms() > 0:
            return mol

    return None

def df_with_valid_ligands(df: pd.DataFrame) -> pd.DataFrame:
    """Drop rows where SDF can't be parsed. Caches RDKit Mol objects in a column."""
    mols = []
    keep = []
    for i, row in df.iterrows():
        mol = read_ligand_from_sdf(row["aligned_ligand_sdf_abs"])
        if mol is None:
            continue
        mols.append(mol)
        keep.append(i)
    out = df.loc[keep].copy().reset_index(drop=True)
    out["_mol"] = mols
    print(f"valid SDF: {len(out)} / {len(df)}")
    return out


## 15. PyTorch dataset wrapping the dataframe

Index = pocket_cluster_id. `__getitem__` returns the list of (pdb_id, Data) for all ligands of one cluster.


In [86]:
class PocketClusterDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, cluster_ids: List):
        self.df = df
        self.cluster_ids = list(cluster_ids)
        self._by_cluster: Dict = {
            cid: df[df["pocket_cluster_id"] == cid].reset_index(drop=True)
            for cid in self.cluster_ids
        }
        # uniprot lookup for hard negatives
        self._cluster_uniprot: Dict = {}
        for cid in self.cluster_ids:
            uids = self._by_cluster[cid]["uniprot_id"].unique()
            # in practice a pocket cluster maps to one uniprot, but be defensive:
            self._cluster_uniprot[cid] = uids[0]
        # reverse index: uniprot -> list of cluster_ids in this split
        self._uniprot_to_clusters: Dict = {}
        for cid, u in self._cluster_uniprot.items():
            self._uniprot_to_clusters.setdefault(u, []).append(cid)

    def __len__(self):
        return len(self.cluster_ids)

    def __getitem__(self, idx):
        cid = self.cluster_ids[idx]
        sub = self._by_cluster[cid]
        items = []
        for _, row in sub.iterrows():
            items.append((row["pdb_id"], row["_mol"]))
        return cid, items

    def hard_negative_clusters(self, anchor_cid, k: int) -> List:
        u = self._cluster_uniprot[anchor_cid]
        siblings = [c for c in self._uniprot_to_clusters.get(u, []) if c != anchor_cid]
        random.shuffle(siblings)
        return siblings[:k]


## 16. Batch sampler

Each training batch:
- Sample B cluster_ids
- For each, split into (1 query, ≥1 positives)
- Optionally augment with up to `n_hard` hard-negative clusters (same uniprot, different cluster) — these go into the batch as additional clusters whose prototypes serve as negatives only (no query drawn from them).


In [87]:
def make_batch(
    dataset: PocketClusterDataset,
    cluster_ids: List,
    n_hard: int = 0,
    rng: Optional[random.Random] = None,
):
    """
    Returns dict:
      'queries':           list of RDKit Mols (length B)
      'positives':         list of list of Mols (length B)    -- one positive set per query
      'hard_neg_groups':   list of list of Mols (length H)    -- extra cluster ligand sets, no queries
      'q_cluster_ids':     list (length B)
      'hn_cluster_ids':    list (length H)
    """
    if rng is None:
        rng = random
    queries, positives, qcids = [], [], []
    for cid in cluster_ids:
        _, items = dataset[dataset.cluster_ids.index(cid)]
        if len(items) < 2:
            continue
        # pick query
        qi = rng.randint(0, len(items) - 1)
        queries.append(items[qi][1])
        positives.append([m for j, (_, m) in enumerate(items) if j != qi])
        qcids.append(cid)

    hard_neg_groups, hncids = [], []
    if n_hard > 0:
        seen = set(qcids)
        for cid in qcids:
            sibs = dataset.hard_negative_clusters(cid, k=n_hard)
            for s in sibs:
                if s in seen:
                    continue
                _, items = dataset[dataset.cluster_ids.index(s)]
                if len(items) < 1:
                    continue
                hard_neg_groups.append([m for (_, m) in items])
                hncids.append(s)
                seen.add(s)
    return {
        "queries": queries,
        "positives": positives,
        "hard_neg_groups": hard_neg_groups,
        "q_cluster_ids": qcids,
        "hn_cluster_ids": hncids,
    }


## 17. Training step

Combined forward:
- Encode **queries** (with grad) via online encoder → `q_proj [B, d_proj]`
- Encode **positives + hard negatives** via key encoder (no grad) → cluster prototypes
  - First B clusters in the prototype tensor are positives (aligned with queries)
  - Next H clusters are hard negatives
- InfoNCE: query i vs prototype i (positive) + all other B+H-1 in-batch prototypes + queue


In [88]:
def encode_mols_with(model: PocketRetrievalModel, mols: List[Chem.Mol], device):
    datas = [mol_to_data(m) for m in mols]
    batch = Batch.from_data_list(datas).to(device)
    return model.encode_ligands(batch)   # [N_mols, d_h]

def encode_prototypes(model: PocketRetrievalModel,
                      ligand_groups: List[List[Chem.Mol]],
                      device) -> torch.Tensor:
    """
    ligand_groups: list of length G, each is list of Mols belonging to one cluster.
    Returns P [G, k, d_h] (key encoder; caller wraps in torch.no_grad if needed).
    """
    flat_mols, group_ids = [], []
    for gi, mols in enumerate(ligand_groups):
        for m in mols:
            flat_mols.append(m)
            group_ids.append(gi)
    z = encode_mols_with(model, flat_mols, device)
    group_ids_t = torch.tensor(group_ids, device=device, dtype=torch.long)
    P, order = model.encode_cluster_from_ligands(z, group_ids_t)
    # order is unique sorted; with group_ids 0..G-1 it equals range(G)
    assert order.tolist() == list(range(len(ligand_groups)))
    return P

def train_step(
    model: PocketRetrievalModel,
    key_model: PocketRetrievalModel,
    queue: MoCoQueue,
    opt: torch.optim.Optimizer,
    batch: Dict,
    cfg: Config,
    device,
):
    # ---- queries (online encoder, with grad)
    q_mols = batch["queries"]
    if len(q_mols) == 0:
        return None
    z_q = encode_mols_with(model, q_mols, device)               # [B, d_h]
    q_proj = model.proj_query(z_q)                              # [B, d_proj]

    # ---- positives + hard negatives (key encoder, no grad)
    all_groups = batch["positives"] + batch["hard_neg_groups"]  # length B + H
    with torch.no_grad():
        P = encode_prototypes(key_model, all_groups, device)    # [B+H, k, d_h]
        P_proj_tokens = key_model.proj_proto(P)                 # [B+H, k, d_proj]
        P_mean = F.normalize(P_proj_tokens.mean(dim=1), dim=-1) # [B+H, d_proj]

    # ---- InfoNCE
    B = len(q_mols)
    Q = queue.get()
    if cfg.proto_collapse == "mean":
        logits_in = q_proj @ P_mean.t()                         # [B, B+H]
        if Q.numel() > 0:
            logits = torch.cat([logits_in, q_proj @ Q.t()], dim=1) / cfg.temperature
        else:
            logits = logits_in / cfg.temperature
    elif cfg.proto_collapse == "max":
        sims = torch.einsum("id,jkd->ijk", q_proj, P_proj_tokens)
        logits_in = sims.max(dim=-1).values                     # [B, B+H]
        if Q.numel() > 0:
            logits = torch.cat([logits_in, q_proj @ Q.t()], dim=1) / cfg.temperature
        else:
            logits = logits_in / cfg.temperature
    else:
        raise ValueError(cfg.proto_collapse)

    labels = torch.arange(B, device=device)                     # positives at index i for query i
    loss = F.cross_entropy(logits, labels)

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    opt.step()

    # ---- EMA update of key encoder
    with torch.no_grad():
        for pq, pk in zip(model.parameters(), key_model.parameters()):
            pk.data.mul_(cfg.momentum).add_(pq.data, alpha=1.0 - cfg.momentum)

    # ---- enqueue positive keys only (mean-collapsed, L2-normed)
    queue.enqueue(P_mean[:B])

    return loss.item()


## 18. Evaluation: Recall@K

Procedure:
- Build a **prototype gallery** over all eval clusters: for each eval cluster, encode its full ligand set with the **key encoder** → mean-collapsed projected prototype `p_c ∈ R^d_proj`.
- For each ligand `L` in eval: encode with online encoder query head, compute similarity to all gallery prototypes, **leave-one-out** by reconstructing its home cluster's prototype from the other ligands.
- Hit@K = positive cluster appears in top-K. Recall@K averaged over all eval anchors.

For efficiency at small/medium eval sizes we rebuild the home prototype on demand per anchor.


In [89]:
@torch.no_grad()
def build_gallery(key_model: PocketRetrievalModel,
                  dataset: PocketClusterDataset,
                  device) -> Tuple[torch.Tensor, List]:
    """Returns (P_gallery [C, d_proj], cluster_ids ordered)."""
    all_groups, cids = [], []
    for cid in dataset.cluster_ids:
        _, items = dataset[dataset.cluster_ids.index(cid)]
        mols = [m for (_, m) in items]
        if len(mols) == 0:
            continue
        all_groups.append(mols)
        cids.append(cid)
    P = encode_prototypes(key_model, all_groups, device)        # [C, k, d_h]
    P_proj = key_model.proj_proto(P)                            # [C, k, d_proj]
    P_mean = F.normalize(P_proj.mean(dim=1), dim=-1)
    return P_mean, cids

@torch.no_grad()
def recall_at_k(model: PocketRetrievalModel,
                key_model: PocketRetrievalModel,
                dataset: PocketClusterDataset,
                ks: List[int],
                device,
                max_anchors_per_cluster: int = 4) -> Dict[int, float]:
    P_gallery, cids = build_gallery(key_model, dataset, device)
    cid_to_idx = {c: i for i, c in enumerate(cids)}

    hits = {k: 0 for k in ks}
    total = 0

    for cid in dataset.cluster_ids:
        _, items = dataset[dataset.cluster_ids.index(cid)]
        mols = [m for (_, m) in items]
        if len(mols) < 2:
            continue
        # pick up to max_anchors per cluster
        rng = random.Random(hash(str(cid)) & 0xFFFF)
        idxs = list(range(len(mols)))
        rng.shuffle(idxs)
        idxs = idxs[:max_anchors_per_cluster]

        for ai in idxs:
            anchor = mols[ai]
            others = [m for j, m in enumerate(mols) if j != ai]

            # leave-one-out home prototype
            P_home = encode_prototypes(key_model, [others], device)   # [1, k, d_h]
            P_home_proj = key_model.proj_proto(P_home)
            p_home_mean = F.normalize(P_home_proj.mean(dim=1), dim=-1)  # [1, d_proj]

            # query
            z_q = encode_mols_with(model, [anchor], device)
            q_proj = model.proj_query(z_q)                              # [1, d_proj]

            # build score vector: replace home position with leave-one-out prototype
            scores = (q_proj @ P_gallery.t()).squeeze(0).clone()        # [C]
            home_idx = cid_to_idx[cid]
            scores[home_idx] = (q_proj @ p_home_mean.t()).squeeze().item()

            ranked = torch.argsort(scores, descending=True).tolist()
            rank = ranked.index(home_idx) + 1
            for k in ks:
                if rank <= k:
                    hits[k] += 1
            total += 1

    return {k: hits[k] / max(total, 1) for k in ks}, total


## 19. End-to-end fit on synthetic CSV (verification)

Since `pdbbind_final_groups.csv` plus the aligned SDFs are not present in this VM, I synthesize a CSV in the exact schema and write fake SDF files to disk, then run the full pipeline (load → split → train → eval).

This verifies all code paths execute and loss decreases.


In [91]:
import tempfile

def synth_pdbbind_csv(out_dir: str,
                      n_uniprots: int = 8,
                      clusters_per_uniprot: int = 2,
                      ligs_per_cluster: Tuple[int,int] = (3,5),
                      year_split: float = 0.85,
                      seed: int = 0):
    """Write fake aligned SDFs + CSV matching pdbbind_final_groups schema."""
    rng = random.Random(seed)
    rows = []
    sdf_dir = Path(out_dir) / "pocket_aligned"
    sdf_dir.mkdir(parents=True, exist_ok=True)
    pdb_counter = 0
    for ui in range(n_uniprots):
        uniprot = f"P{ui:05d}"
        for ci in range(clusters_per_uniprot):
            scaf = SCAFFOLDS[(ui + ci) % len(SCAFFOLDS)]
            M = rng.randint(*ligs_per_cluster)
            pocket_cluster = f"UNIPROT:{uniprot}|SEQCLUST:0|POCKETCLUST:{ci}"
            cluster_dir = sdf_dir / pocket_cluster.replace(":", "_").replace("|", "_")
            cluster_dir.mkdir(exist_ok=True)
            # assign cluster to train (year=2017) or test (year=2019) globally
            year = 2017 if rng.random() < year_split else 2019
            for li in range(M):
                pdb_id = f"{pdb_counter:04x}"
                pdb_counter += 1
                mol = random_analog(scaf, rng)
                mol = Chem.AddHs(mol)
                AllChem.EmbedMolecule(mol, randomSeed=seed + pdb_counter)
                mol = Chem.RemoveHs(mol)
                sdf_path = cluster_dir / f"{pdb_id}_ligand_aligned.sdf"
                w = Chem.SDWriter(str(sdf_path))
                w.write(mol); w.close()
                rows.append({
                    "pdb_id": pdb_id,
                    "release_year": year,
                    "uniprot_id": uniprot,
                    "protein_name": f"PROT_{ui}",
                    "aligned_ligand_sdf": str(sdf_path.relative_to(out_dir)),
                    "pocket_cluster_id": pocket_cluster,
                    "final_status": "ok",
                })
    csv_path = Path(out_dir) / "pdbbind_final_groups.csv"
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    return str(csv_path)

work = tempfile.mkdtemp(prefix="pocket_retrieval_test_")
csv_path = synth_pdbbind_csv(work, n_uniprots=10, clusters_per_uniprot=3,
                             ligs_per_cluster=(3,6), year_split=0.85, seed=42)
print("wrote:", csv_path)
csv_path = "/mnt/data-mol/groupbind/pdbbind_final_groups.csv"
pdbbind_root = "."
df_raw = load_pdbbind_groups(csv_path, pdbbind_root=pdbbind_root)
df_full = df_with_valid_ligands(df_raw)
print(df_full[["pdb_id", "release_year", "uniprot_id", "pocket_cluster_id"]].head())


wrote: /tmp/pocket_retrieval_test_7ss_you3/pdbbind_final_groups.csv
Index(['pdb_id', 'complex_dir', 'protein_pdb', 'ligand_sdf', 'ligand_mol2',
       'release_year', 'uniprot_id', 'protein_name', 'metadata_source',
       'initial_protein_group', 'status', 'sequence_cluster_id',
       'aligned_protein_pdb', 'aligned_ligand_sdf', 'is_reference',
       'alignment_rmsd', 'align_status', 'pocket_cluster_id',
       'min_pl_distance', 'final_status', 'final_group_id'],
      dtype='object')
loaded 16881 rows -> 15655 after filtering
unique pocket_cluster_id: 5580
unique uniprot_id:        1924
valid SDF: 15655 / 15655
  pdb_id  release_year uniprot_id  \
0   10gs          1998     P09211   
1   3gss          1997     P09211   
2   2gss          1997     P09211   
3   3ie3          2009     P09211   
4   1lbk          2002     P09211   

                                   pocket_cluster_id  
0  UNIPROT:P09211|NAME:GLUTATHIONE S-TRANSFERASE ...  
1  UNIPROT:P09211|NAME:GLUTATHIONE S-TRANSF

## 20. Train + evaluate


In [95]:
os.makedirs("models", exist_ok=True)


train_cids, val_cids, test_cids = time_split_clusters(df_full, val_frac=0.15, seed=0)
train_cids = filter_clusters_with_min_size(df_full, train_cids, min_size=2)
val_cids   = filter_clusters_with_min_size(df_full, val_cids,   min_size=2)
test_cids  = filter_clusters_with_min_size(df_full, test_cids,  min_size=1)
print(f"usable: train={len(train_cids)} val={len(val_cids)} test={len(test_cids)}")

train_ds = PocketClusterDataset(df_full, list(train_cids))
val_ds   = PocketClusterDataset(df_full, list(val_cids))
test_ds  = PocketClusterDataset(df_full, list(test_cids))

# --- model
cfg_run = Config(gin_variant="vanilla", proto_collapse="mean",
                 queue_size=256, momentum=0.99, temperature=0.07)
model     = PocketRetrievalModel(cfg_run).to(device)
key_model = copy.deepcopy(model).to(device)
for p in key_model.parameters():
    p.requires_grad_(False)
queue = MoCoQueue(cfg_run.queue_size, cfg_run.d_proj, device)
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# --- training loop
N_EPOCHS = 10
BATCH_CLUSTERS = 8
N_HARD = 2
best_loss = float("inf")
losses = []
val_recalls = []

rng = random.Random(0)
all_train = list(train_ds.cluster_ids)

for ep in range(N_EPOCHS):
    rng.shuffle(all_train)
    ep_losses = []
    for s in range(0, len(all_train), BATCH_CLUSTERS):
        cids = all_train[s:s+BATCH_CLUSTERS]
        batch = make_batch(train_ds, cids, n_hard=N_HARD, rng=rng)
        if len(batch["queries"]) < 2:
            continue
        l = train_step(model, key_model, queue, opt, batch, cfg_run, device)
        if l is not None:
            ep_losses.append(l)
    avg = float(np.mean(ep_losses)) if ep_losses else float("nan")
    losses.append(avg)
    
    if not np.isnan(avg) and avg < best_loss:
        best_loss = avg
        
        torch.save(
            {
                "epoch": ep,
                "model_state_dict": model.state_dict(),
                "key_model_state_dict": key_model.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "loss": best_loss,
                "cfg_run": cfg_run,
            },
            "models/best_model.pt",
        )
        print(f"Saved best_model.pt at epoch {ep}, loss={best_loss:.6f}")

    # val recall every 2 epochs
    if (ep + 1) % 2 == 0 or ep == N_EPOCHS - 1:
        r, n_anchors = recall_at_k(model, key_model, val_ds, ks=[1,3,5], device=device)
        val_recalls.append((ep, r, n_anchors))
        print(f"epoch {ep+1:3d} | loss {avg:.4f} | val R@1={r[1]:.3f} R@3={r[3]:.3f} R@5={r[5]:.3f} (n={n_anchors})")
    else:
        print(f"epoch {ep+1:3d} | loss {avg:.4f}")

# --- test
best_model = torch.load("models/best_model.pt")
test_recall, n_test = recall_at_k(model, key_model, test_ds, ks=[1,3,5,10], device=device)
print(f"\nTEST: R@1={test_recall[1]:.3f}  R@3={test_recall[3]:.3f}  R@5={test_recall[5]:.3f}  R@10={test_recall.get(10, float('nan')):.3f}  (n={n_test})")

assert len(losses) == N_EPOCHS
assert not any(np.isnan(losses)), "loss became nan"
# loss should be lower at end than at start on a learnable synthetic task
print(f"loss[0]={losses[0]:.4f}  loss[-1]={losses[-1]:.4f}  delta={losses[0]-losses[-1]:+.4f}")


clusters: train=4191  val=739  test=435  dropped(mixed)=215
usable: train=2460 val=427 test=435
Saved best_model.pt at epoch 0, loss=4.170378
epoch   1 | loss 4.1704
epoch   2 | loss 4.2766 | val R@1=0.026 R@3=0.080 R@5=0.130 (n=1056)
epoch   3 | loss 4.2358
Saved best_model.pt at epoch 3, loss=4.025010
epoch   4 | loss 4.0250 | val R@1=0.092 R@3=0.242 R@5=0.333 (n=1056)
Saved best_model.pt at epoch 4, loss=3.969662
epoch   5 | loss 3.9697
Saved best_model.pt at epoch 5, loss=3.887125
epoch   6 | loss 3.8871 | val R@1=0.139 R@3=0.295 R@5=0.386 (n=1056)
Saved best_model.pt at epoch 6, loss=3.847901
epoch   7 | loss 3.8479
Saved best_model.pt at epoch 7, loss=3.770428
epoch   8 | loss 3.7704 | val R@1=0.186 R@3=0.348 R@5=0.431 (n=1056)
Saved best_model.pt at epoch 8, loss=3.716253
epoch   9 | loss 3.7163
Saved best_model.pt at epoch 9, loss=3.674878
epoch  10 | loss 3.6749 | val R@1=0.195 R@3=0.370 R@5=0.456 (n=1056)

TEST: R@1=0.210  R@3=0.423  R@5=0.536  R@10=0.667  (n=466)
loss[0]=4.1

In [96]:
train_cids, val_cids, test_cids = time_split_clusters(df_full, val_frac=0.15, seed=0)
train_cids = filter_clusters_with_min_size(df_full, train_cids, min_size=2)
val_cids   = filter_clusters_with_min_size(df_full, val_cids,   min_size=2)
test_cids  = filter_clusters_with_min_size(df_full, test_cids,  min_size=1)
print(f"usable: train={len(train_cids)} val={len(val_cids)} test={len(test_cids)}")

train_ds = PocketClusterDataset(df_full, list(train_cids))
val_ds   = PocketClusterDataset(df_full, list(val_cids))
test_ds  = PocketClusterDataset(df_full, list(test_cids))

# --- model
cfg_run = Config(gin_variant="vnode", proto_collapse="mean",
                 queue_size=256, momentum=0.99, temperature=0.07)
model     = PocketRetrievalModel(cfg_run).to(device)
key_model = copy.deepcopy(model).to(device)
for p in key_model.parameters():
    p.requires_grad_(False)
queue = MoCoQueue(cfg_run.queue_size, cfg_run.d_proj, device)
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# --- training loop
N_EPOCHS = 10
BATCH_CLUSTERS = 8
N_HARD = 2
best_loss = float("inf")
losses = []
val_recalls = []

rng = random.Random(0)
all_train = list(train_ds.cluster_ids)

for ep in range(N_EPOCHS):
    rng.shuffle(all_train)
    ep_losses = []
    for s in range(0, len(all_train), BATCH_CLUSTERS):
        cids = all_train[s:s+BATCH_CLUSTERS]
        batch = make_batch(train_ds, cids, n_hard=N_HARD, rng=rng)
        if len(batch["queries"]) < 2:
            continue
        l = train_step(model, key_model, queue, opt, batch, cfg_run, device)
        if l is not None:
            ep_losses.append(l)
    avg = float(np.mean(ep_losses)) if ep_losses else float("nan")
    losses.append(avg)
    
    if not np.isnan(avg) and avg < best_loss:
        best_loss = avg
        
        torch.save(
            {
                "epoch": ep,
                "model_state_dict": model.state_dict(),
                "key_model_state_dict": key_model.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "loss": best_loss,
                "cfg_run": cfg_run,
            },
            "models/best_model_vnode.pt",
        )
        print(f"Saved best_model.pt at epoch {ep}, loss={best_loss:.6f}")

    # val recall every 2 epochs
    if (ep + 1) % 2 == 0 or ep == N_EPOCHS - 1:
        r, n_anchors = recall_at_k(model, key_model, val_ds, ks=[1,3,5], device=device)
        val_recalls.append((ep, r, n_anchors))
        print(f"epoch {ep+1:3d} | loss {avg:.4f} | val R@1={r[1]:.3f} R@3={r[3]:.3f} R@5={r[5]:.3f} (n={n_anchors})")
    else:
        print(f"epoch {ep+1:3d} | loss {avg:.4f}")

# --- test
best_model = torch.load("models/best_model_vnode.pt")
test_recall, n_test = recall_at_k(model, key_model, test_ds, ks=[1,3,5,10], device=device)
print(f"\nTEST: R@1={test_recall[1]:.3f}  R@3={test_recall[3]:.3f}  R@5={test_recall[5]:.3f}  R@10={test_recall.get(10, float('nan')):.3f}  (n={n_test})")

assert len(losses) == N_EPOCHS
assert not any(np.isnan(losses)), "loss became nan"
# loss should be lower at end than at start on a learnable synthetic task
print(f"loss[0]={losses[0]:.4f}  loss[-1]={losses[-1]:.4f}  delta={losses[0]-losses[-1]:+.4f}")

clusters: train=4191  val=739  test=435  dropped(mixed)=215
usable: train=2460 val=427 test=435
Saved best_model.pt at epoch 0, loss=3.987687
epoch   1 | loss 3.9877
epoch   2 | loss 4.1701 | val R@1=0.030 R@3=0.098 R@5=0.163 (n=1056)
epoch   3 | loss 4.1907
epoch   4 | loss 4.1569 | val R@1=0.062 R@3=0.165 R@5=0.254 (n=1056)
epoch   5 | loss 4.0839
Saved best_model.pt at epoch 5, loss=3.973472
epoch   6 | loss 3.9735 | val R@1=0.114 R@3=0.247 R@5=0.337 (n=1056)
Saved best_model.pt at epoch 6, loss=3.884154
epoch   7 | loss 3.8842
Saved best_model.pt at epoch 7, loss=3.780636
epoch   8 | loss 3.7806 | val R@1=0.154 R@3=0.309 R@5=0.396 (n=1056)
Saved best_model.pt at epoch 8, loss=3.720180
epoch   9 | loss 3.7202
Saved best_model.pt at epoch 9, loss=3.694053
epoch  10 | loss 3.6941 | val R@1=0.198 R@3=0.380 R@5=0.461 (n=1056)

TEST: R@1=0.208  R@3=0.414  R@5=0.543  R@10=0.682  (n=466)
loss[0]=3.9877  loss[-1]=3.6941  delta=+0.2936
